In [ ]:
!pip install pandas requests torch "transformers==4.57.6" accelerate

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import copy
import hashlib
import json
import random

import pandas as pd
import requests

SEED = 42

ROOT = Path("choice_study")
DATA = ROOT / "data"
DATA.mkdir(parents=True, exist_ok=True)

URL = (
    "https://vincentarelbundock.github.io/"
    "Rdatasets/csv/Ecdat/Yogurt.csv"
)

csv_path = DATA / "Yogurt.csv"

if not csv_path.exists():
    response = requests.get(URL, timeout=60)
    response.raise_for_status()
    csv_path.write_bytes(response.content)

raw_bytes = csv_path.read_bytes()
source_hash = hashlib.sha256(raw_bytes).hexdigest()

raw = pd.read_csv(csv_path)

print("Rows:", len(raw))
print("Columns:", raw.columns.tolist())
print("Source SHA256:", source_hash)

display(raw.head(10))

In [ ]:
pd.set_option("display.max_columns", None)

display(raw)
raw.info()

display(raw.isna().sum().rename("missing_values"))
display(raw["choice"].value_counts().rename("purchases"))
display(raw.groupby("id").size().describe().rename("purchases_per_customer"))

In [ ]:
df = raw.rename(columns={
    "rownames": "row_id",
    "id": "household_id",
}).copy()

df["row_id"] = df["row_id"].astype(int)
df["household_id"] = df["household_id"].astype(int)

BRANDS = ["dannon", "hiland", "weight", "yoplait"]
LABELS = dict(zip("ABCD", BRANDS))
BRAND_TO_LABEL = {brand: label for label, brand in LABELS.items()}

household = df["household_id"].iloc[0]

print("All raw purchases belonging to household:", household)
display(df[df["household_id"] == household])

one customer has multiple purchase rows. Randomly splitting individual rows would put the same customer in multiple partitions.

We will instead:

1. Keep customers with at least six observations.
2. Split customer IDs into 60% development, 20% validation and the remainder test.
3. Reserve five observations per customer as support examples.
4. Use that customer’s remaining observations as prediction queries.

In [ ]:
SUPPORT_COUNT = 5

household_counts = df.groupby("household_id").size()

eligible_ids = sorted(
    int(h)
    for h, count in household_counts.items()
    if count > SUPPORT_COUNT
)

excluded_ids = sorted(
    int(h)
    for h, count in household_counts.items()
    if count <= SUPPORT_COUNT
)

shuffled_ids = eligible_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)

n_development = int(0.60 * len(shuffled_ids))
n_validation = int(0.20 * len(shuffled_ids))

household_split = {}

for h in shuffled_ids[:n_development]:
    household_split[h] = "development"

for h in shuffled_ids[n_development:n_development + n_validation]:
    household_split[h] = "validation"

for h in shuffled_ids[n_development + n_validation:]:
    household_split[h] = "test"

observations = df[df["household_id"].isin(eligible_ids)].copy()
observations["split"] = observations["household_id"].map(household_split)
observations["role"] = "query"

for h in eligible_ids:
    rows = observations[
        observations["household_id"] == h
    ].sort_values("row_id")

    # A separate reproducible random selection for each household.
    selected_indices = random.Random(
        f"{SEED}:{h}:support"
    ).sample(rows.index.tolist(), SUPPORT_COUNT)

    observations.loc[selected_indices, "role"] = "support"

queries = observations[observations["role"] == "query"].copy()
support = observations[observations["role"] == "support"].copy()

development = queries[queries["split"] == "development"].copy()
validation = queries[queries["split"] == "validation"].copy()
test = queries[queries["split"] == "test"].copy()

summary = pd.crosstab(observations["split"], observations["role"])
summary["households"] = observations.groupby("split")["household_id"].nunique()

display(summary)
print("Excluded customer IDs:", excluded_ids)

observations[
    ["row_id", "household_id", "split", "role"]
].to_csv(DATA / "row_assignments.csv", index=False)

In [ ]:
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GenerationConfig,
)

MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)

model.eval()

generation_settings = GenerationConfig(
    do_sample=False,
    max_new_tokens=512,
    num_beams=1,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Precision:", DTYPE)

In [ ]:
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
RUN = ROOT / "runs" / run_name
RUN.mkdir(parents=True)

settings = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "seed": SEED,
    "source_sha256": source_hash,
    "thinking": False,
    "explanation_decoding": "greedy",
    "max_explanation_tokens": 512,
}

(RUN / "settings.json").write_text(json.dumps(settings, indent=2))

In [ ]:
query = development.sort_values("row_id").iloc[0]

print("Query row:", int(query["row_id"]))
print("Customer:", int(query["household_id"]))

display(query.to_frame("raw_value"))

In [ ]:
def make_facts(query, with_history):
    facts = []

    for label, brand in LABELS.items():
        facts.append({
            "id": f"P_{label}",
            "type": "price",
            "product": brand,
            "value": float(query[f"price.{brand}"]),
        })

        facts.append({
            "id": f"F_{label}",
            "type": "advertising",
            "product": brand,
            "value": bool(int(query[f"feat.{brand}"])),
        })

    if with_history:
        household_support = support[
            support["household_id"] == query["household_id"]
        ].sort_values("row_id")

        for number, (_, row) in enumerate(
            household_support.iterrows(), start=1
        ):
            facts.append({
                "id": f"H{number}",
                "type": "support",
                "value": {
                    "prices": {
                        brand: float(row[f"price.{brand}"])
                        for brand in BRANDS
                    },
                    "advertised": {
                        brand: bool(int(row[f"feat.{brand}"]))
                        for brand in BRANDS
                    },
                    "purchased": str(row["choice"]),
                },
            })

    return facts

In [ ]:
facts_without_history = make_facts(query, with_history=False)
facts_with_history = make_facts(query, with_history=True)

print("WITHOUT HISTORY")
display(pd.DataFrame(facts_without_history))

print("WITH HISTORY")
display(pd.DataFrame(facts_with_history))

In [ ]:
order = list(LABELS)
random.Random(SEED + int(query["row_id"])).shuffle(order)

def prediction_prompt(facts, order):
    products = [
        {"answer": label, "brand": LABELS[label]}
        for label in order
    ]

    return (
        "Predict which brand was purchased on this purchase occasion.\n"
        "Use the supplied evidence. Customer preferences are unknown.\n"
        "Do not assume the cheapest product must be purchased.\n"
        "Prices retain the original dataset units.\n"
        "Advertising false means not advertised.\n"
        "Support observations are unordered separate purchases.\n"
        "NOT_PROVIDED means unavailable information.\n\n"
        "PRODUCTS:\n"
        + json.dumps(products, indent=2)
        + "\n\nEVIDENCE:\n"
        + json.dumps(facts, indent=2)
        + "\n\nReturn exactly one answer letter: A, B, C, or D."
    )

print(prediction_prompt(facts_without_history, order))
print(prediction_prompt(facts_with_history, order))

In [ ]:
def encode_prompt(text):
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    encoded = tokenizer(
        rendered,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(DEVICE)

    return rendered, encoded


def predict(facts, order):
    text = prediction_prompt(facts, order)
    rendered, encoded = encode_prompt(text)
    prefix = encoded["input_ids"][0].tolist()

    # Identify the answer tokens in the actual prompt context.
    answer_tokens = []

    for label in LABELS:
        full = tokenizer.encode(
            rendered + label,
            add_special_tokens=False,
        )

        suffix = full[len(prefix):]

        if full[:len(prefix)] != prefix or len(suffix) != 1:
            raise ValueError("This scoring method requires one-token answers.")

        answer_tokens.append(suffix[0])

    with torch.inference_mode():
        output = model(
            **encoded,
            use_cache=False,
            logits_to_keep=1,
        )

        answer_logits = output.logits[0, -1, answer_tokens].float()
        answer_scores = torch.softmax(answer_logits, dim=0).cpu().tolist()

    scores = dict(zip(LABELS, answer_scores))

    return {
        "prediction": max(scores, key=scores.get),
        "scores": scores,
        "prompt": text,
        "rendered_prompt": rendered,
        "answer_logits": dict(zip(LABELS, answer_logits.cpu().tolist())),
    }

In [ ]:
development_observations = observations[
    observations["split"] == "development"
]

price_bounds = {
    brand: (
        float(development_observations[f"price.{brand}"].min()),
        float(development_observations[f"price.{brand}"].max()),
    )
    for brand in BRANDS
}


def planned_price_change(facts, predicted_label):
    evidence_id = f"P_{predicted_label}"
    old_price = next(f["value"] for f in facts if f["id"] == evidence_id)

    brand = LABELS[predicted_label]
    low, high = price_bounds[brand]

    # Prefer a 20% increase; otherwise use a feasible 20% decrease.
    for proposed in [old_price * 1.2, old_price * 0.8]:
        new_price = round(proposed, 6)

        if low <= new_price <= high and new_price != old_price:
            return {"evidence_id": evidence_id, "new_value": new_price}

    return None

In [ ]:
def explain(facts, order, prediction, price_change=None):
    # Build a neutral context instead of reusing the prediction instructions.
    products = [
        {"answer": label, "brand": LABELS[label]}
        for label in order
    ]

    available_ids = [fact["id"] for fact in facts]

    text = f"""
Assess the evidence for a previously saved model prediction.

PRODUCTS:
{json.dumps(products, indent=2)}

EVIDENCE:
{json.dumps(facts, indent=2)}

SAVED PREDICTION:
{prediction}: {LABELS[prediction]}

TASK:
Assess whether the supplied evidence provides a defensible reason
for this saved prediction. The prediction may be poorly supported.

The actual purchase for the current query has not been supplied.
Support observations describe separate purchases. Their purchased
brands are not the observed answer to the current query.
Support observations are unordered.

RULES:
1. Describe observable support and limitations for the saved prediction.
2. If no clear supporting evidence exists, say so explicitly.
3. Identify relevant evidence that weakens the saved prediction.
4. Check numerical comparisons against the supplied values.
5. Advertising false means not advertised.
6. NOT_PROVIDED means unavailable information.
7. Do not invent customer motives, attributes, or chronological ordering.
8. Do not claim access to the model's internal reasoning.
9. Keep the explanation to at most three sentences.
10. Discuss only the original supplied evidence and original prediction.

CITATIONS:
Use distinct, exact evidence IDs from:
{json.dumps(available_ids)}

You may cite any number of available IDs.
For factual comparisons, cite the evidence on both sides.
An empty supporting-evidence list is allowed.
Counter-evidence means evidence weakening the prediction;
it does not necessarily rule that product out.

ASSESSMENT:
Choose exactly one:
- "supported": identifiable evidence favors the saved prediction.
- "mixed": relevant evidence both supports and weakens it.
- "insufficient": no defensible supporting reason can be identified.

OUTPUT:
Return exactly one valid JSON object with these keys:
- assessment: one of the three assessment strings above.
- explanation: your concise evidence assessment.
- evidence_ids: IDs supporting the saved prediction.
- counter_evidence_ids: IDs weakening the saved prediction.
- counterfactual_choice: null.

Use double-quoted JSON keys and strings.
Return the JSON object without Markdown fences or surrounding text.
"""

    _, encoded = encode_prompt(text)

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            generation_config=generation_settings,
            use_model_defaults=False,
            do_sample=False,
            num_beams=1,
            temperature=1.0,
            top_p=1.0,
            top_k=50,
            max_new_tokens=512,
        )

    generated = output[0, encoded["input_ids"].shape[-1]:]

    raw_response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return text, raw_response

In [ ]:
results = {}

for condition, with_history in [
    ("no_history", False),
    ("with_history", True),
]:
    facts = make_facts(query, with_history)
    prediction = predict(facts, order)

    price_change = planned_price_change(
        facts, prediction["prediction"]
    )

    explanation_input, raw_explanation = explain(
        facts,
        order,
        prediction["prediction"],
        price_change,
    )

    parsed = None
    explanation_error = None

    try:
        parsed = json.loads(raw_explanation)
        ids = parsed["evidence_ids"]

        valid_ids = (
            isinstance(ids, list)
            and all(isinstance(x, str) for x in ids)
            and len(ids) == len(set(ids))
            and set(ids) <= {f["id"] for f in facts}
        )

        if not valid_ids or not isinstance(parsed["explanation"], str):
            explanation_error = "Invalid explanation fields or evidence IDs"

    except (ValueError, KeyError, TypeError):
        explanation_error = "Response did not match the requested JSON format"

    record = {
        "row_id": int(query["row_id"]),
        "household_id": int(query["household_id"]),
        "condition": condition,
        "facts": facts,
        "order": order,
        **prediction,
        "actual_choice": str(query["choice"]),
        "correct": (
            LABELS[prediction["prediction"]] == query["choice"]
        ),
        "price_change": price_change,
        "explanation_prompt": explanation_input,
        "raw_explanation": raw_explanation,
        "explanation": parsed,
        "explanation_error": explanation_error,
    }

    results[condition] = record

    with (RUN / "original_responses.jsonl").open("a") as file:
        file.write(json.dumps(record, allow_nan=False) + "\n")

    print("\nCONDITION:", condition)
    print("\nPREDICTION INPUT:\n", prediction["prompt"])
    print("\nANSWER SCORES:", prediction["scores"])
    print("PREDICTED BRAND:", LABELS[prediction["prediction"]])
    print("ACTUAL PURCHASE:", query["choice"])
    print("CORRECT:", record["correct"])
    print("\nEXPLANATION INPUT:\n", explanation_input)
    print("\nMODEL EXPLANATION:\n", raw_explanation)
    print("FORMAT ERROR:", explanation_error)

In [ ]:
condition = "with_history"

original = results[condition]
facts = original["facts"]
order = original["order"]
chosen = original["prediction"]

effect_records = []

In [ ]:
def measure_change(name, changed_facts, changed_order):
    after = predict(changed_facts, changed_order)

    effect = {
        "experiment": name,
        "row_id": original["row_id"],
        "household_id": original["household_id"],
        "condition": condition,
        "original_prediction": chosen,
        "edited_prediction": after["prediction"],
        "winner_changed": after["prediction"] != chosen,
        "original_product_score_before": original["scores"][chosen],
        "original_product_score_after": after["scores"][chosen],
        "score_drop": original["scores"][chosen] - after["scores"][chosen],
        "changed_facts": changed_facts,
        "changed_order": changed_order,
        "edited_response": after,
    }

    effect_records.append(effect)

    with (RUN / "counterfactual_responses.jsonl").open("a") as file:
        file.write(json.dumps(effect, allow_nan=False) + "\n")

    print(name)
    print("Winner:", chosen, "→", after["prediction"])
    print(
        "Original product's answer score:",
        round(original["scores"][chosen], 4),
        "→",
        round(after["scores"][chosen], 4),
    )

    return effect

In [ ]:
def mask_values(facts, evidence_ids):
    changed = copy.deepcopy(facts)

    for fact in changed:
        if fact["id"] in evidence_ids:
            fact["value"] = "NOT_PROVIDED"

    return changed

In [ ]:
if original["explanation_error"]:
    print("Cannot evaluate cited evidence:", original["explanation_error"])

else:
    cited_ids = original["explanation"]["evidence_ids"]

    if not cited_ids:
        print("The model supplied no cited evidence.")

    else:
        print("CITED EVIDENCE:")
        display(pd.DataFrame([
            f for f in facts if f["id"] in cited_ids
        ]))

        cited_effect = measure_change(
            "cited_remove",
            mask_values(facts, cited_ids),
            order,
        )

        facts_by_id = {f["id"]: f for f in facts}

        required_types = Counter(
            facts_by_id[eid]["type"] for eid in cited_ids
        )

        pools = {
            kind: [
                f["id"] for f in facts
                if f["type"] == kind and f["id"] not in cited_ids
            ]
            for kind in required_types
        }

        enough_controls = all(
            len(pools[kind]) >= count
            for kind, count in required_types.items()
        )

        controls = []

        if not enough_controls:
            print(
                "No size-and-type-matched uncited comparison is available."
            )

        else:
            rng = random.Random(SEED)
            seen = set()

            for attempt in range(100):
                control_ids = []

                for kind, count in required_types.items():
                    control_ids.extend(
                        rng.sample(pools[kind], count)
                    )

                key = tuple(sorted(control_ids))

                if key in seen:
                    continue

                seen.add(key)

                effect = measure_change(
                    f"uncited_remove_{len(controls) + 1}",
                    mask_values(facts, control_ids),
                    order,
                )

                controls.append(effect)

                if len(controls) == 5:
                    break

            mean_control_drop = sum(
                r["score_drop"] for r in controls
            ) / len(controls)

            advantage = (
                cited_effect["score_drop"] - mean_control_drop
            )

            print("Cited evidence score drop:",
                  cited_effect["score_drop"])
            print("Mean uncited-control score drop:",
                  mean_control_drop)
            print("Cited-minus-control difference:", advantage)

In [ ]:
price_change = original["price_change"]

if price_change is None:
    print("No price change satisfied the predefined development bounds.")

else:
    edited = copy.deepcopy(facts)

    for fact in edited:
        if fact["id"] == price_change["evidence_id"]:
            print(
                "PRICE CHANGE:",
                fact["id"],
                fact["value"],
                "→",
                price_change["new_value"],
            )

            fact["value"] = price_change["new_value"]

    price_effect = measure_change(
        "specified_price_change",
        edited,
        order,
    )

    explanation = original["explanation"]

    expected = (
        explanation.get("counterfactual_choice")
        if isinstance(explanation, dict)
        else None
    )

    print("Model's previously stated counterfactual choice:", expected)
    print("Model's actual edited choice:",
          price_effect["edited_prediction"])

    if expected in LABELS:
        print(
            "Counterfactual prediction matched?",
            expected == price_effect["edited_prediction"],
        )

In [ ]:
changed_product_order = order[1:] + order[:1]

product_order_effect = measure_change(
    "product_order_only",
    copy.deepcopy(facts),
    changed_product_order,
)

In [ ]:
changed_fact_order = list(reversed(copy.deepcopy(facts)))

evidence_order_effect = measure_change(
    "evidence_order_only",
    changed_fact_order,
    order,
)

In [ ]:
display(pd.DataFrame(effect_records)[[
    "experiment",
    "original_prediction",
    "edited_prediction",
    "winner_changed",
    "original_product_score_before",
    "original_product_score_after",
    "score_drop",
]])

In [ ]:
condition = "no_history"
original = results[condition]
facts = original["facts"]
order = original["order"]
chosen = original["prediction"]
effect_records = []

## Joint explanation and final prediction from factual observations

This is a separate, single-case condition. Python computes factual observations from one prepared purchase query. The LLM sees those observations and returns a brief explanation **and** final choice in one response. The two calls below use the same real query, first without support and then with the prepared household support purchases. The target purchase stays hidden until the comparison cell. No prior prediction or explanation enters either prompt.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
from choice_trust.observations import generate_observations, inputs_from_prepared_yogurt
from choice_trust.joint_observation import build_joint_messages, generate_joint_response, save_joint_record

JOINT_CASE_ID = "row-2"  # Stable real prepared case; change only to another case ID in this file.
JOINT_CASES_PATH = Path("data/processed/cases.jsonl")
with JOINT_CASES_PATH.open(encoding="utf-8") as handle:
    joint_record = next((row for line in handle if (row := json.loads(line))["case_id"] == JOINT_CASE_ID), None)
if joint_record is None or joint_record.get("synthetic"):
    raise ValueError(f"Real prepared case {JOINT_CASE_ID!r} not found")
# The allowlist excludes joint_record['target']; only mapped fields can enter the model.
joint_visible = {key: joint_record[key] for key in ("brand_map", "query", "support")}
joint_inputs = {
    "no_history": inputs_from_prepared_yogurt(joint_visible, with_history=False),
    "with_history": inputs_from_prepared_yogurt(joint_visible, with_history=True),
}
joint_results = {}
joint_run_dir = Path("choice_study/joint_observation_runs") / (
    JOINT_CASE_ID + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
print("Prepared case:", JOINT_CASE_ID, "| split:", joint_record["split"])
print("Separate joint-output folder:", joint_run_dir)


### Inspect the source offers and programmed observations

The displayed query offers and support records are the only prepared fields mapped into the generator. Support purchases are separate unordered purchases. The without-support condition uses the same current offers and omits those records.


In [ ]:
print("CANDIDATE MAPPING AND CURRENT OFFERS (both conditions)")
print(json.dumps({"brand_map": joint_visible["brand_map"], "query": joint_visible["query"]}, indent=2))
print("\nSUPPLIED SUPPORT RECORDS (with-history condition only)")
print(json.dumps(joint_visible["support"], indent=2))
joint_observations = {
    condition: generate_observations(**joint_inputs[condition])
    for condition in ("no_history", "with_history")
}
for condition in ("no_history", "with_history"):
    print("\n" + "=" * 16 + " " + condition + " PROGRAMMED OBSERVATIONS " + "=" * 16)
    print(joint_observations[condition]["text"])


### Inspect the complete prompts before inference

The prompt lists answer identifiers, every programmed observation and its ID, and the required JSON response. The Qwen chat template is shown in full for each condition; `enable_thinking=False` matches the existing notebook setting. This section reuses the existing tokenizer if it is already loaded. Otherwise it opens the **same pinned local checkpoint** without downloading a model.


In [ ]:
from transformers import AutoTokenizer
JOINT_MODEL_ID = "Qwen/Qwen3-1.7B"
JOINT_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
JOINT_SNAPSHOT = Path.home() / ".cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots" / JOINT_REVISION
if "tokenizer" not in globals():
    if not JOINT_SNAPSHOT.exists():
        raise RuntimeError("Pinned Qwen tokenizer is not cached; run the existing model-setup cell or restore its checkpoint")
    tokenizer = AutoTokenizer.from_pretrained(str(JOINT_SNAPSHOT), local_files_only=True)
joint_prompt_messages = {
    condition: build_joint_messages(joint_observations[condition], joint_inputs[condition]["candidates"])
    for condition in ("no_history", "with_history")
}
joint_rendered_prompts = {}
for condition in ("no_history", "with_history"):
    joint_rendered_prompts[condition] = tokenizer.apply_chat_template(
        joint_prompt_messages[condition], tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    print("\n" + "=" * 16 + " " + condition + " COMPLETE MESSAGES " + "=" * 16)
    print(json.dumps(joint_prompt_messages[condition], ensure_ascii=False, indent=2))
    print("INCLUDED OBSERVATION IDS:", [o["id"] for o in joint_observations[condition]["observations"]])
    print("COMPLETE RENDERED MODEL PROMPT:")
    print(joint_rendered_prompts[condition])


### One real joint-generation call: without support

This cell reuses the model and `generation_settings` from the existing notebook setup when present. If the old kernel state is absent, it loads the **same pinned checkpoint once from the local cache**. It uses the notebook's greedy 512-token generation protocol and records every effective setting. Re-running this cell creates a new record; no earlier experiment file is overwritten.


In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, GenerationConfig
if "model" in globals():
    if globals().get("MODEL_ID") != JOINT_MODEL_ID or globals().get("MODEL_REVISION") != JOINT_REVISION:
        raise RuntimeError("The loaded model identity does not match the pinned notebook model")
    if "generation_settings" not in globals() or "DEVICE" not in globals() or "DTYPE" not in globals():
        raise RuntimeError("Loaded model is missing its notebook generation/device settings")
else:
    if not JOINT_SNAPSHOT.exists():
        raise RuntimeError("Pinned Qwen model is unavailable locally; inference was not performed")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        str(JOINT_SNAPSHOT), dtype=DTYPE, low_cpu_mem_usage=True,
        local_files_only=True,
    ).to(DEVICE)
    model.eval()
    generation_settings = GenerationConfig(
        do_sample=False, max_new_tokens=512, num_beams=1,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
joint_model_metadata = {
    "model_id": JOINT_MODEL_ID, "revision": JOINT_REVISION,
    "tokenizer": tokenizer.name_or_path, "device": str(DEVICE),
    "dtype": str(DTYPE), "torch": torch.__version__,
    "transformers": transformers.__version__,
}
def show_joint_result(condition):
    result = joint_results[condition]
    parsed = result["parsed_result"] if isinstance(result["parsed_result"], dict) else {}
    print("Exact raw response (repr preserves whitespace):", repr(result["raw_response"]))
    print("Parsed explanation:", parsed.get("explanation"))
    print("Parsed prediction:", parsed.get("prediction"),
          "| product:", result["diagnostics"]["predicted_product"])
    print("Citation and format diagnostics:")
    print(json.dumps(result["diagnostics"], indent=2))
    print("Effective model and generation settings:")
    print(json.dumps(result["generation_metadata"], indent=2))

joint_results["no_history"] = generate_joint_response(
    joint_observations["no_history"], joint_inputs["no_history"]["candidates"],
    model=model, tokenizer=tokenizer, device=DEVICE,
    generation_config=generation_settings, model_metadata=joint_model_metadata,
)
assert joint_results["no_history"]["rendered_prompt"] == joint_rendered_prompts["no_history"]
joint_saved_without = save_joint_record(
    joint_run_dir, case_id=JOINT_CASE_ID, condition="no_history",
    observation_result=joint_observations["no_history"], result=joint_results["no_history"],
)
print("Saved joint record:", joint_saved_without)
show_joint_result("no_history")


### One real joint-generation call: with household support

This separate cell reuses the model already in memory. The current query is unchanged; only the supplied unordered support observations enter the prompt. It writes a second, separate record.


In [ ]:
joint_results["with_history"] = generate_joint_response(
    joint_observations["with_history"], joint_inputs["with_history"]["candidates"],
    model=model, tokenizer=tokenizer, device=DEVICE,
    generation_config=generation_settings, model_metadata=joint_model_metadata,
)
assert joint_results["with_history"]["rendered_prompt"] == joint_rendered_prompts["with_history"]
joint_saved_with = save_joint_record(
    joint_run_dir, case_id=JOINT_CASE_ID, condition="with_history",
    observation_result=joint_observations["with_history"], result=joint_results["with_history"],
)
print("Saved joint record:", joint_saved_with)
show_joint_result("with_history")


### Reveal and compare with the held-out purchase

The target is read **after both model calls** for this one-case display. A valid prediction is compared with the actual prepared purchase; invalid model output stays invalid. This is a case demonstration, not an accuracy estimate.


In [ ]:
import pandas as pd
actual_id = joint_record["target"]  # Kept out of joint_visible, prompts, and saved generation records.
actual_name = joint_visible["brand_map"][actual_id]
comparison_rows = []
for condition in ("no_history", "with_history"):
    result = joint_results[condition]
    parsed = result["parsed_result"] if isinstance(result["parsed_result"], dict) else {}
    predicted_id = parsed.get("prediction") if result["diagnostics"]["prediction_valid"] else None
    cited = parsed.get("evidence_ids") if isinstance(parsed.get("evidence_ids"), list) else []
    by_id = {o["id"]: o["statement"] for o in joint_observations[condition]["observations"]}
    citations = "\n".join(f"{cid}: {by_id.get(cid, 'UNKNOWN ID')}" for cid in cited)
    comparison_rows.append({
        "condition": condition,
        "predicted product": f"{predicted_id}: {joint_visible['brand_map'][predicted_id]}" if predicted_id else "INVALID / NOT_PROVIDED",
        "actual purchased product": f"{actual_id}: {actual_name}",
        "correct for this case": predicted_id == actual_id if predicted_id else "NOT_PROVIDED",
        "exact model explanation": parsed.get("explanation") if isinstance(parsed.get("explanation"), str) else "NOT_PROVIDED",
        "cited observations (full statements)": citations or "None cited",
    })
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(pd.DataFrame(comparison_rows).set_index("condition"))


### Manual inspection checklist

- Did the model describe the current prices and advertising correctly?
- With history, did it state support purchase counts correctly and keep those purchases separate from the hidden target?
- Do the cited observation IDs support its factual statements and comparisons? Are any relevant conflicting observations omitted?
- Did it invent motives, chronology, causal preferences, or unavailable information?
- Did the prediction or explanation change between conditions?

These checks demonstrate execution and help diagnose factual errors. Valid JSON and valid citation IDs do not establish factual correctness or faithfulness. Faithfulness requires later intervention experiments.


## Provisional LLM-assisted claim annotation — one saved explanation

The final cell selects the first saved with-history response matching **both the current compact observation version and the current predictor prompt version**, in sorted saved-file order. If no compatible with-history response exists, it uses the first compatible response without history. It never falls back to legacy prompts, and it does not select based on explanation quality.

The displayed facts are the exact compact input saved with that response, including dollar prices and only P/F/H evidence IDs. The cell annotates the saved explanation using the versioned [annotation guide](docs/EXPLANATION_ANNOTATION_GUIDE.md), without regenerating its prediction or substituting new facts. The target purchase is excluded.

No separate annotator model is configured. The cell reuses the loaded pinned Qwen model, or its existing local cache, and records that predictor and annotator are the same model. Annotation uses greedy decoding with up to 8,192 input and 2,048 output tokens to fit the guide, provenance and claim table. It saves the prompt, raw response, guide snapshot and diagnostics in a separate annotation folder. Re-running this cell makes another annotation attempt, without rerunning prediction. Labels are provisional and require manual review; annotation alone does not establish actual model influence or faithfulness.


In [ ]:
from pathlib import Path
import importlib
import choice_trust.observations as observation_module
importlib.reload(observation_module)
import choice_trust.joint_observation as joint_module
importlib.reload(joint_module)
import choice_trust.explanation_annotation as annotation_module
importlib.reload(annotation_module)  # Refresh selection logic in an already running notebook kernel.
from choice_trust.explanation_annotation import (
    select_saved_explanation, load_guide, cached_local_annotator,
    annotate_explanation, save_annotation,
    display_annotation_input, display_annotation_result,
)

annotation_example = select_saved_explanation(Path("choice_study/joint_observation_runs"))
if annotation_example is None:
    print("No saved explanation matches the current compact observation and predictor-prompt versions. Legacy responses will not be substituted. Run the earlier 'One real joint-generation call: with household support' cell (and its setup cells), then rerun this final cell.")
else:
    print("Selected saved response:", annotation_example["source_path"])
    display_annotation_input(annotation_example)
    annotation_guide = load_guide(Path("docs/EXPLANATION_ANNOTATION_GUIDE.md"))
    try:
        annotation_generate, annotation_model = cached_local_annotator(
            annotation_example["predictor_model"],
            model=globals().get("model"), tokenizer=globals().get("tokenizer"),
        )
    except (RuntimeError, OSError, ImportError) as error:
        print("Annotation inference was not performed:", error)
    else:
        annotation_result = annotate_explanation(
            **{key: annotation_example[key] for key in (
                "explanation", "predicted_candidate", "model_citations", "candidates",
                "observations", "provenance",
            )},
            guide=annotation_guide, generate=annotation_generate,
            annotator_model=annotation_model, predictor_model=annotation_example["predictor_model"],
        )
        annotation_saved_path = save_annotation(
            annotation_result, Path("choice_study/explanation_annotation_runs"), annotation_example,
        )
        print("Saved annotation attempt:", annotation_saved_path)
        display_annotation_result(annotation_result, annotation_example)


## Bounded development counterfactual pilot

This pilot separates **prediction sensitivity** (does the decoded product change?), **factual updating** (does the explanation describe edited facts correctly?), and **claimed-reason consistency** (does behavior agree with an explicit rule or influence claim?). Presentation controls test product order and answer identifiers separately. None of these alone establishes faithfulness or actual customer behavior.

The plan uses two real development households, both support conditions, one response per input, one unchanged rerun per case/condition, at most 32 joint calls and six annotation calls. Every edit is selected from source facts before inference. The next cells save and show the plan before loading a model. The output directory is versioned; rerunning a completed stage reuses only exact-compatible records.


In [ ]:
from pathlib import Path
from choice_trust.pilot_execution import Pilot, load_runtime, DEFAULT_OUT
from choice_trust.pilot_notebook import (
    show_cases, show_manifest, show_originals, show_edits, show_annotations, show_results,
)
pilot = Pilot.prepare(DEFAULT_OUT, seed=42)
print("Saved plan:", pilot.out / "manifest.json")
print("Predeclared budgets:", pilot.manifest["budgets"])


### Selected real cases and original facts

Selection uses seed 42 and development-data availability, preferring informative historical price and advertising variation. The cases come from distinct households. The same query is shown without support and with the supplied unordered support purchases; query targets are excluded.


In [ ]:
show_cases(pilot)


### Manifest before inference

Inspect the exact planned patches and skips. Each registry entry defines permitted fields, fixed fields, selection rules and dependent summaries. Counts and historical offer relationships remain within HOUSEHOLD_HISTORY. The historical association searches permute complete offer vectors and remove equivalent unordered assignments; they do not invent offer variation.


In [ ]:
show_manifest(pilot)


### Four original joint responses

Reuse the configured model already in memory when available, otherwise load its pinned local cache once. Every stage shares this runtime. The joint predictor produces an explanation and final answer in one call, with the same 512-token greedy settings throughout the pilot. Existing notebook prediction/annotation outputs are not reused as fresh baselines. Available exact-compatible annotations are shown; the selective annotation stage below fills planned annotation slots.


In [ ]:
pilot_runtime = load_runtime(model=globals().get("model"), tokenizer=globals().get("tokenizer"))
pilot.attach(pilot_runtime)
pilot.run("ORIGINAL")
show_originals(pilot)


### Unchanged-input sanity reruns

One fresh unchanged-input rerun per original case and condition. These have distinct call IDs, so they are not replaced by baseline cache hits. One rerun is a sanity check, not an estimate of an answer distribution.


In [ ]:
pilot.run("UNCHANGED_RERUN")
show_edits(pilot, "UNCHANGED_RERUN")


### Current price edits

The first case preserves its complete cheapest set with a one-cent edit. The second makes a previously noncheapest candidate uniquely cheapest. Candidate/value selection is fixed before inference and ignores model predictions and target purchases. All positive cent-valued edits lie inside the saved development-only bounds. Full source changes and regenerated facts are displayed below.


In [ ]:
pilot.run("CURRENT_PRICE")
show_edits(pilot, "CURRENT_PRICE")


### Current advertising edits

Toggle the first candidate with a known flag. Keep all prices and household history fixed. Missing flags remain unavailable, rather than becoming false.


In [ ]:
pilot.run("CURRENT_ADVERTISING")
show_edits(pilot, "CURRENT_ADVERTISING")


### Support purchase edits

Change the first support purchase to the next valid candidate. Historical prices and advertisements stay fixed. This household-history edit changes purchase counts and purchase–offer relationships together; they are dependent summaries, not independent concepts.


In [ ]:
pilot.run("SUPPORT_PURCHASE")
show_edits(pilot, "SUPPORT_PURCHASE")


### Historical price-association edits

Keep purchases, counts, historical flags and current offers fixed. Reassign complete historical price vectors to change how often purchased brands are among the cheapest. The manifest records the feasible unique assignments, selected donors, and before/after eligible counts. Without-history inputs receive an explicit skip.


In [ ]:
pilot.run("HISTORICAL_PRICE_ASSOCIATION")
show_edits(pilot, "HISTORICAL_PRICE_ASSOCIATION")


### Historical advertising-association edits

Keep purchases, counts, historical prices and current offers fixed. Reassign complete historical advertising vectors to change purchased-advertised counts. Cases with constant flags or invariant relationships are skipped rather than given invented variation.


In [ ]:
pilot.run("HISTORICAL_ADVERTISING_ASSOCIATION")
show_edits(pilot, "HISTORICAL_ADVERTISING_ASSOCIATION")


### Historical offer removal

Keep support IDs and purchased products. Replace all historical prices and advertising with null/NOT_PROVIDED. The generator recomputes every dependent offer comparison and denominator; missing offers are never encoded as zero prices or false flags.


In [ ]:
pilot.run("HISTORICAL_OFFER_REMOVAL")
show_edits(pilot, "HISTORICAL_OFFER_REMOVAL")


### Product-order and answer-mapping controls

The product-order control changes presentation order while retaining each answer identifier. The answer-mapping control consistently relabels candidates, current offers, support purchases and historical offers. Compare **decoded product identities**, not letters. Both controls regenerate complete prompts and verify semantic product information before inference.


In [ ]:
pilot.run(["PRODUCT_ORDER_CONTROL", "ANSWER_MAPPING_CONTROL"])
show_edits(pilot, ["PRODUCT_ORDER_CONTROL", "ANSWER_MAPPING_CONTROL"])


### Selective annotation and before/after reason comparison

The predeclared six slots cover four originals and one edited response per case (historical-price association when informative, otherwise price reversal). The existing versioned guide and same cached model are used. Annotation receives only that response and its own facts/provenance, with no target purchase or paired outcome. Invalid outputs remain visible. Annotations are provisional; remaining edited explanations stay not yet annotated.

Compare claimed groups, roles, candidate direction and factual statuses against the displayed before/after facts. Do not infer an explicit deciding rule from a price mention. Inspect quotation and mapping diagnostics before trusting a claimed-reason comparison.


In [ ]:
pilot.run_annotations()
show_annotations(pilot)
show_originals(pilot)
show_edits(pilot, ["HISTORICAL_PRICE_ASSOCIATION", "CURRENT_PRICE"])


### Measured results and limits

The following table is built only from saved measured responses, skips, and available provisional annotations. Unreviewed explanations receive no invented factual or reason-consistency verdict. It reports raw case counts, not statistical significance or a universal trust score.


In [ ]:
show_results(pilot)


### Actual measured results and interpretation

- Completed **32 joint calls** and **6 annotation calls**, with 0 execution failures. All four unchanged reruns reproduced the exact original raw response.
- Decoded product changes among measured edits: CURRENT_PRICE: 4/4; CURRENT_ADVERTISING: 3/4; SUPPORT_PURCHASE: 1/2; HISTORICAL_PRICE_ASSOCIATION: 1/2; HISTORICAL_ADVERTISING_ASSOCIATION: 1/2; HISTORICAL_OFFER_REMOVAL: 1/2; PRODUCT_ORDER_CONTROL: 2/4; ANSWER_MAPPING_CONTROL: 1/4. Eight household-history interventions were skipped because history was NOT_PROVIDED.
- Price edits changed all four predictions, including both household-53 conditions where the cheapest brand stayed fixed. Several explanations contain factual errors: Yoplait $5.30 called cheapest despite Hiland $4.90; all prices called equal after the household-99 reversal; H5 treated as most recent despite unordered history. Exact manual spot-check quotes and references are saved separately.
- Annotation results: 6/6 JSON parses, 2/6 structural/traceability passes, 0/6 truncations. These annotations remain provisional and require semantic/coverage review. The predictor and annotator share the same checkpoint. The first annotation incorrectly marks a false cheapest-price claim as supported and omits an explicit deciding-rule statement; its quote also fails validation. See the separate limited `annotation_review.json`. Remaining edited responses are not yet annotated.

An unchanged answer does not establish zero influence; other evidence can offset a contributing reason. A changed explanation can correctly update the facts. An explicit cheapest-brand deciding rule has stronger behavioral implications than price merely contributing; the original household-53 no-history response already gets the cheapest brand wrong, limiting any rule-consistency conclusion. Presentation comparisons use decoded products. One response per input and one unchanged rerun do not estimate reliable answer distributions. Actual purchases under edited scenarios are unknown. This development pilot does not establish customer counterfactual behavior or conclusive faithfulness.

See [the actual results and interpretation](choice_study/development_counterfactual_v1/pilot_summary.md) and [the protocol](docs/DEVELOPMENT_COUNTERFACTUAL_PILOT.md). Resume using `.venv/bin/python scripts/run_counterfactual_pilot.py all`; the completed plan is signature-verified and reused without model loading or repeated calls.


## Test split: frozen 100-case with-history study — read-only results

This section reads the saved `choice_study/test_with_history_100_v1/` snapshot. It never loads a model, submits inference, annotates, retries, or changes saved records. Run these cells again to refresh local results. If execution is on EC2, explicitly synchronize the remote study directory first using the handoff instructions; loading a local snapshot does not fetch remote progress.

The original 70 cells and their outputs are preserved. No unchanged-input reruns or repeated sampling are part of this stage. Edited-scenario customer purchases are unknown, so only original-input prediction accuracy is reported.


In [ ]:
from pathlib import Path
import sys

study_results_root = Path.cwd()
if not (study_results_root / "choice_trust").is_dir() and (study_results_root.parent / "choice_trust").is_dir():
    study_results_root = study_results_root.parent
if str(study_results_root) not in sys.path:
    sys.path.insert(0, str(study_results_root))
from choice_trust.test100_results import load_saved_results, show_overview, show_families, show_items, show_examples

test100_saved = load_saved_results(study_results_root / "choice_study" / "test_with_history_100_v1")
show_overview(test100_saved)


### Measured completion, applicability and prediction changes

All eight predefined families remain visible, including absent, pending and not-applicable measurements. A prediction-change rate uses only original/edited pairs with valid parsed predictions and decoded product identities. Each row also reports the applicable denominator, skipped reasons, failures, uncertain items and unassessable pairs. A model-output parsing failure is separate from an execution failure.


In [ ]:
show_families(test100_saved)


### Saved product identities, parsing and review status

Predictions are compared after decoding answer labels to canonical product identity. Citation validity and parsing status are displayed separately from prediction changes. No factual correctness or faithfulness labels are assigned by this loader. Bulk annotations have not been launched; unchecked pilot annotations are not evaluation labels for this study.


In [ ]:
show_items(test100_saved)


### Actual original/edited explanations and exact edits

The examples below use the first available saved pair in each family, in fixed family order, up to four examples. They are selected by availability, not correctness, answer changes or interesting explanations. Expand each record to inspect exact facts, prompts, raw output, original citations, generation settings and diagnostics. Malformed responses are retained.


In [ ]:
show_examples(test100_saved, limit=4)


### Interpretation limits

- Completion is determined from saved item states and response presence. Pending, failed and uncertain work is unresolved; the study is not complete while required work remains.
- Original accuracy is shown as correct / assessable predictions with known targets. Correct / all selected cases is also shown to expose incomplete-run coverage. Read both denominators before interpreting partial results.
- Edited-scenario accuracy is undefined because actual purchases under the edits are unavailable. Product changes measure response sensitivity, not customer correctness or proven faithfulness.
- Explanation text changes are not validated changes in reasoning. Claimed influence, factual correctness, claim coverage and human-review status require separate annotation and review.
- The saved pilot annotations and agent-assisted review remain separate. Correcting six pilot annotations does not validate the annotator for this 100-case study.


In [ ]:
#[REDACTED_API_KEY]

## Completed test metrics — saved records only

This section reads `choice_study/test_with_history_100_v1/analysis_v1/metrics.json`. It does not run prediction, inference, annotation, or the analysis runner. Earlier cells and outputs are preserved.

**Reference-label assumption:** existing explanation annotations are assumed correct, without semantic re-review or a human-review approval gate. No human validation is claimed. **There are zero annotations linked to this 100-case run.** The saved pilot/older annotations refer to other cases and are excluded by scope, so annotation-dependent rates are NA rather than invented labels or zero error rates.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, HTML
from html import escape

metrics_directory = Path("choice_study/test_with_history_100_v1/analysis_v1")
if not metrics_directory.exists() and (Path("..") / metrics_directory).exists():
    metrics_directory = Path("..") / metrics_directory
saved_metrics = json.loads((metrics_directory / "metrics.json").read_text())
saved_metric_examples = json.loads((metrics_directory / "examples.json").read_text())
def saved_rate(value):
    return "NA (0/0)" if value["rate"] is None else f"{value['percentage']:.2f}% ({value['numerator']}/{value['denominator']})"
accuracy = saved_metrics["original_accuracy"]
display(Markdown(f"**Original accuracy: {saved_rate(accuracy)} — {accuracy['numerator']} correct out of {accuracy['denominator']}.** Coverage: {accuracy['evaluated_originals']}/100 selected cases. Each original is counted once."))
display(pd.DataFrame([saved_metrics["data_coverage"]]))
display(Markdown(saved_metrics["annotation_reference_assumption"]))


### Prediction changes and presentation invariance

A change means that the edited response selects a different **decoded canonical product**, after applying its answer mapping. The denominator contains valid original–edited prediction pairs. Skips and malformed responses are not unchanged predictions. Invariance is the unchanged-product fraction for the same control pairs. No edited-scenario customer accuracy is defined because actual edited purchases are unknown.


In [ ]:
prediction_rows = []
for row in saved_metrics["prediction_change_by_family"]:
    prediction_rows.append({"family": row["family"], "applicable": row["applicable"], "completed/reused": row["completed"] + row["reused"], "assessable": row["assessable"], "changed": row["changed"], "unchanged": row["unchanged"], "change rate": saved_rate(row["change_rate"]), "skips": row["skipped"], "failed": row["failed"], "unresolved": row["unresolved"], "completed but unassessable": row["completed_unassessable"]})
display(pd.DataFrame(prediction_rows))
display(pd.DataFrame([{"control": family, "invariance": saved_rate(value)} for family, value in saved_metrics["invariance"].items()]))


### Claimed influence and conditional change rates

The two conditional rates compare change among pairs with and without influence **asserted in the original explanation reference labels**. “No influence asserted” does not establish that the model ignored the evidence. Decisive/contributing claims remain claimed influence even when their factual premise is false. History subtypes are not inferred from HOUSEHOLD_HISTORY alone; offer removal retains purchase frequency. Presentation controls are separate.

The table includes the 2×2 counts and excluded unknown labels. The overall row is **pair-weighted**: several interventions share an original, so pairs are not independent observations. Missing annotations are excluded, not assigned to “no influence.”


In [ ]:
conditional_rows = []
for row in saved_metrics["conditional_change_by_family"] + [saved_metrics["conditional_change_overall_pair_weighted"]]:
    conditional_rows.append({"family": row["family"], "valid prediction pairs": row["valid_prediction_pairs"], "yes / changed": row["yes_changed"], "yes / unchanged": row["yes_unchanged"], "no / changed": row["no_changed"], "no / unchanged": row["no_unchanged"], "P(change | claimed influence)": saved_rate(row["p_change_given_claimed_influence"]), "P(change | no influence asserted)": saved_rate(row["p_change_given_no_influence_asserted"]), "unclear excluded": row["unclear_excluded"], "unassessable excluded": row["unassessable_excluded"], "invalid prediction pairs excluded": row["invalid_prediction_pairs_excluded"]})
display(pd.DataFrame(conditional_rows))
display(pd.DataFrame([saved_metrics["annotation_coverage"]]))


### Rule consistency, factual contradictions and stale facts

Rule consistency requires an explicit sufficiently specified original rule, valid premises, a unique rule-implied edited product and a decodable edited prediction. A contributing reason or “more likely” phrase is not a unique-product rule.

Factual contradiction rate is contradicted / (supported + contradicted) **annotated factual claims**. Unsupported, unclear and not-applicable statuses remain separate. Originals are counted once, even when multiple interventions reference them.

A stale fact must assert an original fact that the edit changed or removed as still applicable. A new unrelated error, a legitimate past/original reference, or changed reasoning is not automatically stale. Existing claim mappings/temporal labels are required. With no linked annotations, all these rates are NA. Invalid-premise and ambiguous-rule counts are unknown, not assumed zero.


In [ ]:
quality_rows = []
rule_overall = saved_metrics["rule_consistency"]["OVERALL_PAIR_WEIGHTED"]
quality_rows.append({"metric": "Rule consistency", "rate": saved_rate(rule_overall["rule_consistency_rate"]), "assessment coverage": f"{rule_overall['assessable_pairs']}/{rule_overall['applicable_pairs']} substantive pairs", "reason": "Missing original rule annotations"})
for scope in ("originals", "edited_overall", "all_distinct_responses"):
    row = saved_metrics["factual_contradiction"][scope]
    quality_rows.append({"metric": "Factual contradiction: " + scope, "rate": saved_rate(row["contradiction_rate"]), "assessment coverage": f"{row['annotated_response_count']}/{row['response_count']} responses", "reason": row["reason"]})
stale_overall = saved_metrics["stale_fact"]["OVERALL_EDITED_RESPONSES"]
quality_rows.append({"metric": "Stale fact", "rate": saved_rate(stale_overall["stale_fact_rate"]), "assessment coverage": f"{stale_overall['assessable_edited_explanations']}/{stale_overall['edited_response_count']} edited responses", "reason": "Missing edited claim annotations, stale labels and temporal/asserted-value mappings"})
display(pd.DataFrame(quality_rows))
display(pd.DataFrame([{"scope": scope, **saved_metrics["factual_contradiction"][scope]["claim_status_counts"], "missing annotated responses": saved_metrics["factual_contradiction"][scope]["missing_annotation_count"]} for scope in ("originals", "edited_overall", "all_distinct_responses")]))
display(Markdown("These zero claim counts mean **no reference claims are available**, not zero factual errors. Per-family rates, coverage and exclusion reasons are in metrics.json and the evidence files."))


### Actual saved examples and traceable outputs

Examples illustrate measured prediction changes, unchanged predictions, remapping invariance and a malformed edited response. The examples use saved text and exact patches; **no new explanation labels are assigned**. Annotation-based categories have no examples because this run has no linked annotations.


In [ ]:
for example in saved_metric_examples:
    display(Markdown(f"**{example['category']} — {example['case_id']} / {example['family']}**. Decoded canonical product: {example['original_product_id']} → {example['edited_product_id']}"))
    display(pd.DataFrame([{"input": "original", "response ID": example["original_response_id"], "exact explanation": example["original_explanation"]}, {"input": "edited", "response ID": example["edited_response_id"], "exact explanation": example["edited_explanation"]}]))
    detail = {key: example[key] for key in ("exact_patch", "original_raw_response", "edited_raw_response", "annotation_status")}
    display(HTML("<details><summary>Exact edit, raw responses and annotation status</summary><pre>" + escape(json.dumps(detail, ensure_ascii=False, indent=2)) + "</pre></details>"))


### Reproduce and inspect

Run `python3 scripts/analyze_test100.py` from the repository to recalculate from saved records. Loading these notebook cells only reads the saved analysis files.

[Markdown report](choice_study/test_with_history_100_v1/analysis_v1/report.md) · [metrics.json](choice_study/test_with_history_100_v1/analysis_v1/metrics.json) · [per-pair table](choice_study/test_with_history_100_v1/analysis_v1/pair_analysis.csv) · [annotation inventory](choice_study/test_with_history_100_v1/analysis_v1/annotation_inventory.json).

Claim, rule and stale-fact evidence files include explicit inclusion/exclusion reasons and stable response/annotation IDs. The empty claim-level file reflects absent labels; all 742 responses remain in the coverage records. Source hashes and consistency checks preserve provenance. No KL-based CE, distribution-based EE, Pearson faithfulness, combined trust score or counterfactual customer accuracy is computed.


## GPT-6 Luna — inspect ten original explanations

This new stage prepares exactly **10 distinct ORIGINAL with-history explanations** from the completed100-case test run. It uses `gpt-6-luna` to extract claims, concept groups, claimed influence, factual judgments and explicit rule candidates. It does not regenerate predictions, annotate edited responses or continue to the other90 cases.

**Execution status: 10/10 real Luna annotation responses saved; 9/10 pass structural/traceability checks.** The API calls were executed after the user supplied a key and authorized the run. This covers only these ten originals, with zero edited-response annotations. Rerunning matching completed cells reuses records; failed/uncertain attempts are not repeated automatically. Run only this new section, not the earlier predictor cells.

Model-generated judgments remain reviewable. Exact-quote/schema checks verify structure and traceability, not semantic correctness, claim coverage or faithfulness. A model coverage self-report is not human review. No human-reviewed status is assigned automatically.


In [ ]:
from pathlib import Path
import sys
import json

luna_project_root = Path.cwd()
if not (luna_project_root / "choice_trust").is_dir() and (luna_project_root.parent / "choice_trust").is_dir():
    luna_project_root = luna_project_root.parent
if str(luna_project_root) not in sys.path:
    sys.path.insert(0, str(luna_project_root))
from choice_trust.luna_annotations import LunaInspection
from choice_trust.luna_views import show_configuration, show_selected, show_prompt, show_source, show_annotation, show_summary

luna_inspection = LunaInspection.prepare()
show_configuration(luna_inspection)
print("Prepared directory:", luna_inspection.out)


### Selected ten originals

Selection uses seed42 over the100 original case IDs sorted numerically, without examining correctness or explanation quality. The displayed source hashes identify the exact response being annotated. Targets, correctness, edited outcomes and earlier annotation judgments are excluded from annotator inputs.


In [ ]:
show_selected(luna_inspection)


### Full prompt preview — no key or API call

Inspect the complete request for the first frozen case, including the original guide, versioned supplement, exact facts/explanation/citations, evidence-field mapping and strict JSON schema. The schema supports only CURRENT_PRICES, CURRENT_ADVERTISING and HOUSEHOLD_HISTORY as parent groups. Rules retain exact quotes, scope, conditions, explicitness, factual-premise status and potential later-test eligibility.


In [ ]:
show_prompt(luna_inspection)


### First real API annotation — run explicitly after supplying the key

Add `OPENAI_API_KEY` privately to the ignored project `.env`, or set it in this kernel's environment. **Do not put the key in a notebook cell or print it.** The empty `.env.example` template shows the variable name. If this kernel lacks dependencies, run `%pip install -r requirements-annotation.txt` first.

**This cell can make one real API call.** Missing credentials return instructions without creating an attempt. It requests the exact Luna model, high reasoning, strict Structured Outputs and a32768-token reasoning/output allowance. It never falls back to another model. Matching completed records are reused; failed/uncertain attempts are not automatically repeated.


In [ ]:
# API CELL — intentionally unexecuted during implementation.
show_source(luna_inspection)
luna_first_result = luna_inspection.annotate_first()
show_annotation(luna_inspection, luna_first_result)


### Remaining nine — run only after inspecting the first result

This is a separate explicit execution step. It processes only the other nine frozen original jobs, reuses matching saved records, and never adds edited inputs or the remaining test cases. Inspect the first result's raw text, all important claims, rules and evidence values before deciding to run this cell. Doing so does not automatically mark anything human-reviewed.


In [ ]:
# API CELL — intentionally unexecuted during implementation.
luna_remaining_results = luna_inspection.annotate_remaining()
if isinstance(luna_remaining_results, dict):
    print(json.dumps(luna_remaining_results, ensure_ascii=False, indent=2))
show_summary(luna_inspection)


### Read-only inspection summary

This table loads saved state/results only. Before execution, claim/rule counts and factual labels are unavailable rather than fabricated empty answers. Afterwards it shows every selected case's status, model coverage self-report, factual labels, rule eligibility and structural diagnostics. Ten original annotations provide at most10/100 original coverage and **zero edited-response coverage**. No counterfactual rule-consistency or stale-fact assessment is generated here.


In [ ]:
show_summary(luna_inspection)


### Inspection checklist and saved records

- Were important factual assertions or explicit deciding rules missed?
- Were false minimum-price, support-count, recency or actual-paid-price claims incorrectly called supported?
- Was a description confused with influence, or a contributing reason promoted to a decisive rule?
- Did a vague trade-off become an invented universal rule?
- Do evidence IDs and specific source fields support/evaluate the quoted claim, including comparisons across groups?

**Execute in order:** configuration → selected examples → prompt preview → first API cell → inspect its result → remaining-nine API cell → summary. The first three steps and summary need no key. The API cells were not executed during implementation.

Canonical records, raw API responses, attempts and diagnostics will be saved under `choice_study/test_with_history_100_v1/luna_originals_10_v1/`. Complete structurally traceable claim projections are saved separately under `annotations/luna-originals-10-v1/` for existing metrics joins. Full rule objects remain separate for later rule-aware analysis; no synthetic stale labels or full-study metrics are populated.

[Workflow and official documentation](docs/LUNA_ORIGINALS_10.md). The existing annotation guide, previous outputs and `analysis_v1` remain unchanged.
